# Imports


In [1]:
import numpy as np
import torch
import time
import copy
import torch
import rendering
import heightfield
import image
import opt
import sys
import simannealing

In [2]:
print("Running with " + str(sys.version))

Running with 3.8.8 (default, Apr 13 2021, 12:59:45) 
[Clang 10.0.0 ]


# Grad descent

In [7]:
def grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, sim_annealing = True, optim_type = "Adam", heaviside_fn = None, heaviside_k = 0.1, num_repeats = 2, num_rays = 32):
  
  num_cams = len(cam_directions)
  num_strips = H.shape[0]*num_repeats
  ray_width = 1/num_rays
  
  original_ray_origins = [[], [], [], []]
  ray_origins = copy.deepcopy(original_ray_origins)
  coords = []
  
  schedule = torch.zeros(50) #update colors for 10 steps, then heights for 20 steps
  schedule[10:20] = 1
  schedule[40:50] = 1

  grad_types = schedule.repeat(20)
  

  for i in range(num_rays):
    for k in range(num_rays):
      coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
      coords.append(coord)

      for q in range(num_cams):   
        curr_origin = coord - 20*cam_directions[q]
        curr_origin[2] = curr_origin[2].item()
        ray_origins[q].append(curr_origin)



  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)



  ray_packs = []


  for t in range(num_cams):
    ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))

  
  desired_cols = []
  for i in range(num_cams):
    desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))
  
  
  H, COL = simannealing.sim_annealing(H, COL, ray_packs, num_cams, num_repeats, heaviside_fn, heaviside_k, desired_cols)
      
  if optim_type == "Adam":
    optimizer = torch.optim.AdamW([H, COL],lr=0.1)
    
  if optim_type == "SGD":
    optimizer = torch.optim.SGD([H, COL],lr=0.1, momentum=0.9)


  for j in range(1, num_epochs):
    
    torch.save(H, 'march20H.pt')
    torch.save(COL, 'march20COL.pt')
      
    change = False
    
    if j%100 == 0:
      H = heightfield.double_resolution(H)
      COL = heightfield.double_resolution(COL)
      H = torch.tensor(H, dtype=torch.float)
      COL = torch.tensor(COL, dtype=torch.float)
      num_strips = num_strips*2
      
      desired_cols = []
      for q in range(num_cams):

        desired_cols.append(image.get_tensor_from_image(image_paths[q], (64, 64), (num_rays, num_rays)))
      change = True


    if j%100 == 95:
      num_rays = num_rays*2
      ray_width = ray_width/2
      change = True
    
    if j%100 == 95 or j%100 == 0:
      coords = []
      change = True
      

      
      ray_origins = copy.deepcopy(original_ray_origins)
      for i in range(num_rays):
        for k in range(num_rays):
          coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
          coords.append(coord)
          for q in range(num_cams):    
            curr_origin = coord - 20*cam_directions[q] 
            curr_origin[2] = curr_origin[2].item()
            ray_origins[q].append(curr_origin)


      
      ray_packs = []
      for t in range(num_cams):
        ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))
        
      H.requires_grad = True
      COL.requires_grad = True
      
      desired_cols = []
      for i in range(num_cams):
        desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))

    if int(grad_types[j].item()) == 0:
      
      H.requires_grad = False
      COL.requires_grad = True
      

      lr = 0.05
      

    elif int(grad_types[j].item()) == 1:
      H.requires_grad = True
      COL.requires_grad = False 
      COL = COL.detach().numpy()
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = False)
      if j > 49:
        lr = 0.03
      
    elif int(grad_types[j].item()) == 2:
      H.requires_grad = True
      COL.requires_grad = True
      H = torch.tensor(H, dtype=torch.float, requires_grad = True)
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)

    if j > 0:
      if int(grad_types[j].item()) != int(grad_types[j-1].item()):
        
        change = True
    if change:



      if optim_type == "Adam":
        optimizer = torch.optim.AdamW([H, COL],lr=lr)
      if optim_type == "SGD":
        optimizer = torch.optim.SGD([H, COL],lr=lr, momentum=0.9)




    result_cols = rendering.forward(H, COL, ray_packs, num_cams, num_repeats=num_repeats, heaviside_fn = heaviside_fn, heaviside_k = heaviside_k)

    mse_loss = 0
    for q in range(num_cams):
      mse_loss += torch.nn.functional.mse_loss(result_cols[q].double(), desired_cols[q].double())




    if j % 1 == 0 or j == 99:
      print("EPOCH: " + str(j) + "\t LOSS: " + str(mse_loss.item()/2))
      time.sleep(1)
    if use_barrier_loss:
      barrier_loss = opt.barrier_fn(H, min_val = 0.1, max_val = 1.5)
      mse_loss = mse_loss + 0.02*barrier_loss

    if use_neighbor_loss:
      neighbor_loss = opt.neighbor_fn(H)
      mse_loss = mse_loss + 0.5*neighbor_loss



      
       
    
    optimizer.zero_grad()
    mse_loss.backward()
    optimizer.step()



  repeated_H = H.repeat(num_repeats, num_repeats)
  repeated_COL = COL.repeat(num_repeats, num_repeats, 1)


  return repeated_H, repeated_COL, result_cols, desired_cols



# Main

In [8]:
#azimuth compatibility ablation

H = heightfield.get_uniform_H(4, 4, center =0.85)
COL = heightfield.get_white_COL(4)

cam_direction_1 = np.array([1, 0.000001, -1])

cos = [1, 0.94, 0.766, 0.5, 0.174, -0.174, -0.5, -0.766, -0.94, -1]
sin = [0.001, 0.34, 0.643, 0.866, 0.984, 0.984, 0.866, 0.643, 0.34, 0.001]

np.set_printoptions(threshold=1000)
torch.set_printoptions(threshold = 1000)

cam_direction_1 = np.array([1, 0.000001, -2])
cam_direction_2 = np.array([-1, 0.000001, -2])
cam_direction_3 = np.array([0.000001, 1, -2])
cam_directions = np.array([cam_direction_1, cam_direction_2, cam_direction_3])

num_epochs = 260
p1 = "test_images/green.png"
p2 = "test_images/red.png"
p3 = "test_images/blue.png"
image_paths = [p1, p2, p3]
H1, COL1, result_cols, desired_cols = grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, optim_type="Adam", heaviside_fn = "tan", num_rays = 16)




<ipython-input-7-1e795c81fe92>:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
<ipython-input-7-1e795c81fe92>:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)


STEP: 0	 LOSS: 1.7631373965344344
STEP: 1	 LOSS: 1.7481410613262174
STEP: 2	 LOSS: 1.7420800400674028
STEP: 3	 LOSS: 1.7249437858020151
STEP: 4	 LOSS: 1.6965130372613344
STEP: 5	 LOSS: 1.6870231648498295
STEP: 6	 LOSS: 1.6796116089795743
STEP: 7	 LOSS: 1.6713295715566736
STEP: 8	 LOSS: 1.6520829294983033
STEP: 9	 LOSS: 1.652305243767601
STEP: 10	 LOSS: 1.6557789599330586
STEP: 11	 LOSS: 1.6570627622417695
STEP: 12	 LOSS: 1.64890468752346
STEP: 13	 LOSS: 1.650830267006036
STEP: 14	 LOSS: 1.632780593779351
STEP: 15	 LOSS: 1.638322173321286
STEP: 16	 LOSS: 1.6298406599925204
STEP: 17	 LOSS: 1.6271682114736497
STEP: 18	 LOSS: 1.620241884467767
STEP: 19	 LOSS: 1.6240976126010147
STEP: 20	 LOSS: 1.6271291995487474
STEP: 21	 LOSS: 1.628231772502875
STEP: 22	 LOSS: 1.6269893290783783
STEP: 23	 LOSS: 1.6144303530437623
STEP: 24	 LOSS: 1.608567538037824
STEP: 25	 LOSS: 1.5954329186115066
STEP: 26	 LOSS: 1.5845951021873352
STEP: 27	 LOSS: 1.5800968374803488
STEP: 28	 LOSS: 1.5672098207666956
STEP

<ipython-input-7-1e795c81fe92>:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float)
<ipython-input-7-1e795c81fe92>:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float)


EPOCH: 100	 LOSS: 0.1608014665156734
EPOCH: 101	 LOSS: 0.1511187303901846
EPOCH: 102	 LOSS: 0.14478084526498608
EPOCH: 103	 LOSS: 0.14096401961372493
EPOCH: 104	 LOSS: 0.13856388688444074
EPOCH: 105	 LOSS: 0.13699000659580432
EPOCH: 106	 LOSS: 0.1362167990334312
EPOCH: 107	 LOSS: 0.13614085920300728
EPOCH: 108	 LOSS: 0.13645294986595138
EPOCH: 109	 LOSS: 0.13678131802129403
EPOCH: 110	 LOSS: 0.13695092439705794
EPOCH: 111	 LOSS: 0.1292432400292095
EPOCH: 112	 LOSS: 0.1253318412202231
EPOCH: 113	 LOSS: 0.12769877567348295
EPOCH: 114	 LOSS: 0.12614098904578094
EPOCH: 115	 LOSS: 0.1210145302259866
EPOCH: 116	 LOSS: 0.11905495876250399
EPOCH: 117	 LOSS: 0.11739410822019997
EPOCH: 118	 LOSS: 0.11763926484786441
EPOCH: 119	 LOSS: 0.11730909871442793
EPOCH: 120	 LOSS: 0.12137554160737446
EPOCH: 121	 LOSS: 0.11547074692683226
EPOCH: 122	 LOSS: 0.11250658986072545
EPOCH: 123	 LOSS: 0.1113722327084482
EPOCH: 124	 LOSS: 0.11068256839510482
EPOCH: 125	 LOSS: 0.10986551269414643
EPOCH: 126	 LOSS: 0